# EdgeConBCE — PLM family on a larger ESM-2 checkpoint

The published PLM results use `esm2_t6_8M_UR50D`, the smallest released ESM-2
checkpoint, which bounds every conclusion about the PLM family. This notebook
re-runs the same 10-seed protocol on a larger checkpoint and regenerates the
manuscript's numbers, tables and figures from the new artifacts.

Nothing in the analysis code changes: the checkpoint is selected through the
`EDGECONBCE_ESM_MODEL` environment variable, and the encoder depth is derived
from the checkpoint name.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

Expected runtime on a T4: roughly 1.5-3 hours for `esm2_t33_650M_UR50D` over 10
seeds, dominated by the ESM forward passes. Feature caching means a re-run is
much faster. Colab disconnects idle sessions, so keep the tab active.

### Where the time goes

The GPU is needed only to extract frozen ESM-2 features, once per checkpoint;
those are cached under `outputs/feature_cache/<checkpoint>/`. Everything after
that is CPU work, and it dominates: in the published 10-seed BP3 runs the edge
tree ensemble took about 330 s of each 332 s seed, single-threaded, for roughly
77 minutes in total.

This notebook sets `EDGECONBCE_N_JOBS=-1`, which fits that ensemble on all
available cores. Forest fitting is deterministic given `random_state`, so the
numbers are unchanged — verified bit-identical — and only the wall clock moves.
Pick the runtime with the most vCPUs you can get; the GPU tier is close to
irrelevant here.

In [ ]:
# --- what to run -------------------------------------------------------------
# Leave ESM_MODEL as None to pick automatically from the GPU's memory.
ESM_MODEL = None            # e.g. "esm2_t33_650M_UR50D" or "esm2_t12_35M_UR50D"
SEEDS_OVERRIDE = None       # e.g. [42, 43, 44] for a shorter pilot run
RUN_NESTED_FUSION = True    # regenerate the leakage-free fusion analysis too

# --- where the project comes from --------------------------------------------
REPO_URL = ""                                                   # option A
DRIVE_PROJECT_ZIP = "/content/drive/MyDrive/edgeconbce_project.zip"  # option B

from pathlib import Path
PROJECT_DIR = Path("/content/EdgeConBCE")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/edgeconbce_outputs")

CHECKPOINT_BASE_URL = "https://dl.fbaipublicfiles.com/fair-esm/models"
REGRESSION_BASE_URL = "https://dl.fbaipublicfiles.com/fair-esm/regression"
print("configuration set")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Set Runtime -> Change runtime type -> T4 GPU, then rerun."
    )

name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
import os
cores = os.cpu_count()
print(f"GPU: {name} ({total_gb:.1f} GiB)")
print(f"vCPUs: {cores}")

# The seed sweep is dominated by a CPU tree ensemble, not by the GPU: in the
# published 10-seed BP3 runs the edge refiner took ~330 s of each ~332 s seed.
# Worker count therefore sets the wall clock far more than the GPU tier does.
os.environ["EDGECONBCE_N_JOBS"] = "-1"
est = 77 * 2 / max(cores, 1)
print(f"\nEstimated 10-seed fitting time at {cores} workers: ~{est:.0f} min "
      f"(~77 min single-threaded).")
if cores <= 2:
    print("This runtime has few vCPUs. A high-CPU machine finishes the same job "
          "several times faster; the GPU choice barely matters.")

# Checkpoint sizing. The 650M encoder needs roughly 12 GiB of headroom for the
# contact head on long antigens; the 35M variant is the fallback for small cards.
if ESM_MODEL is None:
    if total_gb >= 14:
        ESM_MODEL = "esm2_t33_650M_UR50D"
    elif total_gb >= 7:
        ESM_MODEL = "esm2_t12_35M_UR50D"
    else:
        ESM_MODEL = "esm2_t6_8M_UR50D"
    print(f"auto-selected checkpoint: {ESM_MODEL}")
else:
    print(f"using requested checkpoint: {ESM_MODEL}")

if ESM_MODEL == "esm2_t33_650M_UR50D" and total_gb < 14:
    raise RuntimeError(
        f"{ESM_MODEL} needs about 14 GiB; this GPU has {total_gb:.1f} GiB. "
        "Use esm2_t12_35M_UR50D instead."
    )

In [ ]:
DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    DRIVE_AVAILABLE = True
except Exception as exc:
    print("Drive mount unavailable:", repr(exc))
    DRIVE_OUTPUT_DIR = Path("/content/edgeconbce_outputs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs will be copied to", DRIVE_OUTPUT_DIR)

In [ ]:
import subprocess, zipfile, shutil

if PROJECT_DIR.exists() and (PROJECT_DIR / "pyproject.toml").exists():
    print("using existing project at", PROJECT_DIR)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
elif Path(DRIVE_PROJECT_ZIP).exists():
    with zipfile.ZipFile(DRIVE_PROJECT_ZIP) as archive:
        archive.extractall(PROJECT_DIR.parent)
    if not (PROJECT_DIR / "pyproject.toml").exists():
        candidates = [p for p in PROJECT_DIR.parent.iterdir() if (p / "pyproject.toml").exists()]
        if len(candidates) != 1:
            raise RuntimeError(f"Cannot identify the project root among {candidates}")
        if PROJECT_DIR.exists():
            shutil.rmtree(PROJECT_DIR)
        candidates[0].rename(PROJECT_DIR)
else:
    raise RuntimeError("Set REPO_URL, or upload the project zip to DRIVE_PROJECT_ZIP.")

assert (PROJECT_DIR / "pyproject.toml").exists(), "project root not found"
print("project ready at", PROJECT_DIR)

In [ ]:
%cd /content/EdgeConBCE
!python -m pip install -q --upgrade pip
!python -m pip install -q -e . fair-esm scikit-learn pyyaml scipy matplotlib

In [ ]:
import urllib.request
from pathlib import Path

ckpt_dir = PROJECT_DIR / ".torchcache" / "hub" / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

for url, target in (
    (f"{CHECKPOINT_BASE_URL}/{ESM_MODEL}.pt", ckpt_dir / f"{ESM_MODEL}.pt"),
    (f"{REGRESSION_BASE_URL}/{ESM_MODEL}-contact-regression.pt",
     ckpt_dir / f"{ESM_MODEL}-contact-regression.pt"),
):
    if target.exists() and target.stat().st_size > 0:
        print("present:", target.name)
        continue
    print("downloading", target.name)
    urllib.request.urlretrieve(url, target)
    print(f"  {target.stat().st_size / 1024**2:.1f} MiB")

print("checkpoints ready")

In [ ]:
import os, yaml, json
from pathlib import Path

os.environ["EDGECONBCE_ESM_MODEL"] = ESM_MODEL

# Write a run configuration whose output directory names the checkpoint, so
# runs at different model sizes never overwrite one another.
base = yaml.safe_load(Path("configs/bp3_repeated_eval_sci.yaml").read_text())
run_tag = ESM_MODEL.replace("esm2_", "").replace("_UR50D", "")
base["output_dir"] = f"outputs/bp3_repeated_eval_{run_tag}"
base["run_name"] = f"bp3_real_multiseed_{run_tag}"
if SEEDS_OVERRIDE:
    base["seeds"] = list(SEEDS_OVERRIDE)

config_path = Path(f"configs/bp3_repeated_eval_{run_tag}.yaml")
config_path.write_text(yaml.safe_dump(base, sort_keys=False))
print(config_path.read_text())

In [ ]:
# The long run. Progress is written to the output directory as each seed
# finishes, so a disconnect loses at most the seed in flight.
!EDGECONBCE_ESM_MODEL=$ESM_MODEL EDGECONBCE_N_JOBS=-1 python -m edgeconbce.pipeline.run --config $config_path

In [ ]:
import json
from pathlib import Path

run_tag = ESM_MODEL.replace("esm2_", "").replace("_UR50D", "")
out_dir = Path(f"outputs/bp3_repeated_eval_{run_tag}")
aggregate = out_dir / f"bp3_real_multiseed_{run_tag}_aggregate_results.json"
assert aggregate.exists(), f"missing {aggregate}; the run did not finish"

d = json.loads(aggregate.read_text())
print("seeds:", d["seeds"])
print("checkpoint recorded:", d.get("modern_extension", {}).get("plm_model"))
print()
print(f"{'policy':<32}{'MCC':>8}{'ROC-AUC':>10}{'latency ms':>12}")
for row in d["aggregate_policy_metrics"]:
    print(f"{row['policy']:<32}{row['mcc_mean']:>8.4f}{row['roc_auc_mean']:>10.4f}"
          f"{row['mean_latency_ms_mean']:>12.2f}")

## Regenerate the manuscript artifacts

Every number, table and figure in the paper is produced by script from the
result files, so pointing the generators at the new aggregate file is all that
is needed. Review the numbers before replacing the committed versions — the
manuscript's prose makes claims about the PLM family that a larger checkpoint
may change, and those sentences must be rewritten to match.

In [ ]:
import subprocess, shlex
from pathlib import Path

run_tag = ESM_MODEL.replace("esm2_", "").replace("_UR50D", "")
agg = f"outputs/bp3_repeated_eval_{run_tag}/bp3_real_multiseed_{run_tag}_aggregate_results.json"
paper = Path("paper")

if RUN_NESTED_FUSION:
    for cmd in [
        f"python -m edgeconbce.pipeline.nested_fusion --dataset bp3 "
        f"--carbon-predictions outputs/carbon_bce_r_colab_bp3/carbon_bce_r_test_predictions.json "
        f"--bepipred-raw-scores outputs/external_baselines/bepipred_submission/bepipred3_results/raw_output.csv "
        f"--reference-threshold 0.1512 --output-dir outputs/nested_fusion_{run_tag}",
    ]:
        print("$", cmd); subprocess.run(shlex.split(cmd), check=True)

for cmd in [
    f"python tools/make_numbers.py --repeated-eval {agg} "
    f"--bepipred-summary outputs/external_baselines/bepipred_submission/evaluation/bepipred3_local_rerun_summary.json "
    f"--replication outputs/replication_check.json "
    f"--nested-bp3 outputs/nested_fusion_{run_tag}/nested_fusion_bp3_mcc.json "
    f"--nested-iedb outputs/nested_fusion_sci/nested_fusion_iedb_mcc.json "
    f"--output paper/generated_numbers_{run_tag}.tex",
    f"python tools/make_figures.py --repeated-eval {agg} "
    f"--nested-bp3 outputs/nested_fusion_{run_tag}/nested_fusion_bp3_mcc.json "
    f"--nested-iedb outputs/nested_fusion_sci/nested_fusion_iedb_mcc.json "
    f"--output-dir paper/figures_{run_tag}",
]:
    print("$", cmd)
    subprocess.run(shlex.split(cmd), check=False)

In [ ]:
import shutil, datetime
from pathlib import Path

run_tag = ESM_MODEL.replace("esm2_", "").replace("_UR50D", "")
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
archive = Path(f"/content/edgeconbce_{run_tag}_{stamp}")
shutil.make_archive(str(archive), "zip", root_dir="outputs",
                    base_dir=f"bp3_repeated_eval_{run_tag}")
target = DRIVE_OUTPUT_DIR / f"{archive.name}.zip"
shutil.copy(f"{archive}.zip", target)
print("saved", target, f"({target.stat().st_size / 1024**2:.1f} MiB)")
print()
print("Next: copy this archive into the repository's outputs/ directory, rerun")
print("paper/build.sh, and update the PLM sentences in the manuscript to match.")